# Real-world Data Wrangling

In this project, you will apply the skills you acquired in the course to gather and wrangle real-world data with two datasets of your choice.

You will retrieve and extract the data, assess the data programmatically and visually, accross elements of data quality and structure, and implement a cleaning strategy for the data. You will then store the updated data into your selected database/data store, combine the data, and answer a research question with the datasets.

Throughout the process, you are expected to:

1. Explain your decisions towards methods used for gathering, assessing, cleaning, storing, and answering the research question
2. Write code comments so your code is more readable

## 1. Gather data

In this section, you will extract data using two different data gathering methods and combine the data. Use at least two different types of data-gathering methods.

### **1.1.** Problem Statement
This project is motivated by the idea that pet ownership may support personal well-being and that homes with pets may be less attractive to potential burglars. Because direct daily pet-ownership data are unavailable, the analysis will examine whether Google search interest in common pet products -- a limited proxy for pet-related activity -- is  associated with reported burglary and breaking-and-entering offenses in North Carolina from 2021 through 2025.

The first dataset will contain daily Google Trends results collected with Pytrends for pet-product terms such as food, beds, kennels, treats, and collars. The second dataset consists of yearly FBI NIBRS CSV files merged and filtered for North Carolina property offenses.

### **1.2.** Gather at least two datasets using two different data gathering methods

List of data gathering methods:

- Download data manually
- Programmatically downloading files
- Gather data by accessing APIs
- Gather and extract data from HTML files using BeautifulSoup
- Extract data from a SQL database

Each dataset must have at least two variables, and have greater than 500 data samples within each dataset.

For each dataset, briefly describe why you picked the dataset and the gathering method (2-3 full sentences), including the names and significance of the variables in the dataset. Show your work (e.g., if using an API to download the data, please include a snippet of your code).

Load the dataset programmtically into this notebook.

#### Dataset 1

Type: Time-series data loaded into a pandas DataFrame and saved as a CSV file

Method: The data was gathered using the Gather data by accessing APIs method through Pytrends, an unofficial interface for Google Trends. I selected this dataset because it measures changes in North Carolina search interest for products commonly associated with pet ownership from 2021 through 2025.

Dataset variables:

*   date: Date associated with each daily Google Trends observation
*   kennel: Adjusted relative Google search interest for 'kennel' within the Pets & Animals category
*   food: Adjusted relative Google search interest for 'food' within the Pets & Animals category
*   bed: Adjusted relative Google search interest for 'bed' within the Pets & Animals category
*   treats: Adjusted relative Google search interest for 'treats' within the Pets & Animals category
*   collar: Adjusted relative Google search interest for 'collar' within the Pets & Animals category

In [137]:
import pandas as pd
from pytrends.request import TrendReq
from pathlib import Path

Access Google search trends using pytrends.

* Each raw Google Trends request returns relative interest scores from 0 - 100.
* Overlapping periods are rescaled to create one comparable 5-year index.

In [138]:
pytrends = TrendReq(hl='en-US', tz = 240) # Eastern time zone in the United States

In [118]:

search_terms = ['kennel', 'food', 'bed', 'treats', 'collar']


In [140]:
# Create a function that automates the steps for retrieving trend data
def get_trends_period(start_date, end_date):
    start_text = pd.Timestamp(start_date).strftime('%Y-%m-%d')
    end_text = pd.Timestamp(end_date).strftime('%Y-%m-%d')

    pytrends.build_payload(
        search_terms,
        cat = 66,
        timeframe = f'{start_text} {end_text}',
        geo = 'US-NC',
        gprop = ''
    )

    period_data = pytrends.interest_over_time()

    period_data = period_data.drop(columns='isPartial', errors='ignore')

    return period_data

In [143]:
# Create a function that finds shared dates between periods, compares average scores on those dates, calculates a scale factor, scales the newest period, removes overlapping dates and stacks newest dates to the existing data
def scale_and_combine(existing_data, new_period):
    overlap_dates = existing_data.index.intersection(new_period.index)

    existing_overlap_average = (existing_data.loc[overlap_dates, search_terms].mean(axis=1))

    new_overlap_average = (new_period.loc[overlap_dates, search_terms].mean(axis=1))

    scale_factor = (existing_overlap_average.sum() / new_overlap_average.sum())

    new_period_scaled = new_period * scale_factor

    new_dates = new_period_scaled[new_period_scaled.index > existing_data.index.max()]

    combined_data = pd.concat([existing_data, new_dates])

    return combined_data

In [148]:
# Define start and end dates of product interest trends collection
project_start = pd.Timestamp('2021-01-01')
project_end = pd.Timestamp('2025-12-31')
# Request 89 days of data at a time
window_days = 89
# Overlap by 7 days, used to determine scale factor
overlap_days = 7


In [149]:
# Calculate the first window's start and end dates
current_start = project_start
current_end = min(current_start + pd.Timedelta(days=window_days - 1), project_end)
current_start, current_end

(Timestamp('2021-01-01 00:00:00'), Timestamp('2021-03-30 00:00:00'))

In [150]:
# Retrieve trend data for the first 89 days
daily_trends = get_trends_period(current_start, current_end)

In [152]:
import time

# Continue until the dataset reaches the final project data
while current_end < project_end:
    # Begin the next period 7 days before the current period ends
    next_start = current_end - pd.Timedelta(days=overlap_days - 1)

    # Calculate the end of the next 89 day period
    # min() prevents the date from exceeding project_end date
    next_end = min(next_start + pd.Timedelta(days=window_days - 1), project_end)

    # Get the next period from Google Trends
    new_period = get_trends_period(next_start, next_end)

    # Scale the new period and stack it
    daily_trends = scale_and_combine(daily_trends, new_period)

    # Show progress
    print(f'Added data through {next_end.date()}: {daily_trends.shape[0]} total rows')

    # Move the current ending date forward
    current_start = next_start
    current_end = next_end

    # Pause to avoid sending requests too quickly
    time.sleep(10)


Added data through 2021-06-20: 171 total rows
Added data through 2021-09-10: 253 total rows
Added data through 2021-12-01: 335 total rows
Added data through 2022-02-21: 417 total rows
Added data through 2022-05-14: 499 total rows
Added data through 2022-08-04: 581 total rows
Added data through 2022-10-25: 663 total rows
Added data through 2023-01-15: 745 total rows
Added data through 2023-04-07: 827 total rows
Added data through 2023-06-28: 909 total rows
Added data through 2023-09-18: 991 total rows
Added data through 2023-12-09: 1073 total rows
Added data through 2024-02-29: 1155 total rows
Added data through 2024-05-21: 1237 total rows
Added data through 2024-08-11: 1319 total rows
Added data through 2024-11-01: 1401 total rows
Added data through 2025-01-22: 1483 total rows
Added data through 2025-04-14: 1565 total rows
Added data through 2025-07-05: 1647 total rows
Added data through 2025-09-25: 1729 total rows
Added data through 2025-12-16: 1811 total rows
Added data through 2025-

In [153]:
daily_trends.index.min(), daily_trends.index.max(), daily_trends.index.duplicated().sum(), daily_trends.isna().sum()

(Timestamp('2021-01-01 00:00:00'),
 Timestamp('2025-12-31 00:00:00'),
 np.int64(0),
 kennel    0
 food      0
 bed       0
 treats    0
 collar    0
 dtype: int64)

In [154]:
# Save the raw Google trends dataset for later use
daily_trends.to_csv('NC_pet_product_trends_2021_2025.csv')

#### **Dataset 2**

Type: Multiple CSV files combined into a pandas DataFrame

Method: The data was gathered using the Download Data Manually method from the FBI Crime Data Explorer at https://cde.ucr.cjis.gov/LATEST/webapp/#/pages/downloads. I selected this dataset because it contains detailed records that can be used to measure reported property offenses for the years 2021-2025.

Dataset variables:

*   incident_date: Date on which the reported incident occurred
*   offense_code: FBI code identifying the type of offense
*   offense_name: Readable name of the reported offense
*   offense_category_name: General category assigned to the offense
*   county_name: North Carolina county associated with the reporting agency
*   crime_against: Classification indicating whether the offense was against property, a person, or society

In [69]:
from zipfile import ZipFile
import os
from pathlib import Path


In [70]:
# Create a function that reads and merges columns the FBI csv files needed for this investigation
# The FBI database provides a separate download for each year

def prepare_nibrs_year(folder_path):
    folder_path = Path(folder_path)

    incidents = pd.read_csv(folder_path / "NIBRS_incident.csv",
                            usecols = [
                                'data_year',
                                'nibrs_month_id',
                                'incident_id',
                                'agency_id',
                                'incident_date'
                            ])
    offenses = pd.read_csv(folder_path / "NIBRS_OFFENSE.csv",
                           usecols = [
                               'data_year',
                               'incident_id',
                               'offense_id',
                               'offense_code'
                           ])
    offense_types = pd.read_csv(folder_path / "NIBRS_OFFENSE_TYPE.csv",
                                usecols = [
                                    'offense_code',
                                    'offense_name',
                                    'crime_against',
                                    'offense_category_name'
                                ])
    agencies = pd.read_csv(folder_path / "agencies.csv",
                         usecols = [
                             'data_year',
                             'agency_id',
                             'state_postal_abbr',
                             'county_name'
                         ])
    # Combine offenses with incidents on keys data_year and incident_id
    # Note: one incident_id has the potential to appear in more than one yearly file, so incident_id and data_year together is more accurate for exploration
    combined_data = offenses.merge(incidents,
                                   on = ['data_year', 'incident_id'],
                                   how = 'inner', # keep offenses that have a matching incident
                                   validate = 'many_to_one') # one incident can have many offenses
    # Now combine with offense_types to add readable crime category
    combined_data = combined_data.merge(offense_types,
                                        on = 'offense_code',
                                        how = 'left', # keep all offense rows, even those without offense_codes
                                        validate = 'many_to_one') # many offenses with have the same offense_code, each with one definition
    # Now combine with agencies to add state and county
    combined_data = combined_data.merge(agencies,
                                        on = ['data_year', 'agency_id'],
                                        how = 'left', # keep all offense rows
                                        validate = 'many_to_one') # many offense rows from one agency
    # Filter the table to keep data from NC and property crimes
    # .copy() creates a DataFrame separate from the original combined_data
    combined_data = combined_data[
        (combined_data['state_postal_abbr'] == 'NC') &
        (combined_data['crime_against'] == 'Property')].copy()
    # Return the final dataset combined_data
    return combined_data

In [71]:
years = [2021, 2022, 2023, 2024, 2025] # years to be investigated
crime_by_year = {} # empty dictionary for storing yearly crime DataFrames

In [72]:
# Extract each yearly zip file
for yr in years:
    zip_path = Path(fr'C:\Users\steff\Downloads\NC-{yr}.zip')
    extract_folder = Path('NC_crime_data') / str(yr)

    with ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_folder)

    print(f'Extracted {yr}')

Extracted 2021
Extracted 2022
Extracted 2023
Extracted 2024
Extracted 2025


In [73]:
# Find folders containing each year's CSV files
data_folders = {}

for yr in years:
    extract_folder = Path('NC_crime_data') / str(yr)

    incident_file = next(extract_folder.rglob('NIBRS_incident.csv'))

    data_folders[yr] = incident_file.parent # returns folder containing the incident file and saves in data_folders dictionary

    print(yr, data_folders[yr])

2021 NC_crime_data\2021
2022 NC_crime_data\2022
2023 NC_crime_data\2023\NC
2024 NC_crime_data\2024
2025 NC_crime_data\2025


In [74]:
# Create a DataFrame for each year
for yr in years:
    crime_by_year[yr] = prepare_nibrs_year(data_folders[yr])
    print(yr, crime_by_year[yr].shape)

2021 (337746, 12)
2022 (345497, 12)
2023 (343890, 12)
2024 (331271, 12)
2025 (307154, 12)


In [75]:
# Concatenate the 5 separate DataFrames into one DataFrame
crime_data = pd.concat(crime_by_year.values(), ignore_index = True)
crime_data.shape

(1665558, 12)

In [76]:
crime_data['data_year'].value_counts().sort_index()

data_year
2021    337746
2022    345497
2023    343890
2024    331271
2025    307154
Name: count, dtype: int64

In [77]:
# Save the DataFrame to a csv to eliminate reprocessing
crime_data.to_csv('NC_property_crime_2021_2025.csv', index = False)

Finding the right datasets can be time-consuming. Here we provide you with a list of websites to start with. But we encourage you to explore more websites and find the data that interests you.

* Google Dataset Search https://datasetsearch.research.google.com/
* The U.S. Government’s open data https://data.gov/
* UCI Machine Learning Repository https://archive.ics.uci.edu/ml/index.php


## 2. Assess data

Assess the data according to data quality and tidiness metrics using the report below.

List **two** data quality issues and **two** tidiness issues. Assess each data issue visually **and** programmatically, then briefly describe the issue you find.  **Make sure you include justifications for the methods you use for the assessment.**

### Quality Issue 1:

In [ ]:
#FILL IN - Inspecting the dataframe visually

In [ ]:
#FILL IN - Inspecting the dataframe programmatically

Issue and justification: *FILL IN*

### Quality Issue 2:

In [ ]:
#FILL IN - Inspecting the dataframe visually

In [ ]:
#FILL IN - Inspecting the dataframe programmatically

Issue and justification: *FILL IN*

### Tidiness Issue 1:

In [ ]:
#FILL IN - Inspecting the dataframe visually

In [ ]:
#FILL IN - Inspecting the dataframe programmatically

Issue and justification: *FILL IN*

### Tidiness Issue 2: 

In [ ]:
#FILL IN - Inspecting the dataframe visually

In [ ]:
#FILL IN - Inspecting the dataframe programmatically

Issue and justification: *FILL IN*

## 3. Clean data
Clean the data to solve the 4 issues corresponding to data quality and tidiness found in the assessing step. **Make sure you include justifications for your cleaning decisions.**

After the cleaning for each issue, please use **either** the visually or programatical method to validate the cleaning was succesful.

At this stage, you are also expected to remove variables that are unnecessary for your analysis and combine your datasets. Depending on your datasets, you may choose to perform variable combination and elimination before or after the cleaning stage. Your dataset must have **at least** 4 variables after combining the data.

In [ ]:
# FILL IN - Make copies of the datasets to ensure the raw dataframes 
# are not impacted

### **Quality Issue 1: FILL IN**

In [ ]:
# FILL IN - Apply the cleaning strategy

In [ ]:
# FILL IN - Validate the cleaning was successful

Justification: *FILL IN*

### **Quality Issue 2: FILL IN**

In [ ]:
#FILL IN - Apply the cleaning strategy

In [ ]:
#FILL IN - Validate the cleaning was successful

Justification: *FILL IN*

### **Tidiness Issue 1: FILL IN**

In [ ]:
#FILL IN - Apply the cleaning strategy

In [ ]:
#FILL IN - Validate the cleaning was successful

Justification: *FILL IN*

### **Tidiness Issue 2: FILL IN**

In [1]:
#FILL IN - Apply the cleaning strategy

In [2]:
#FILL IN - Validate the cleaning was successful

Justification: *FILL IN*

### **Remove unnecessary variables and combine datasets**

Depending on the datasets, you can also peform the combination before the cleaning steps.

In [ ]:
#FILL IN - Remove unnecessary variables and combine datasets

## 4. Update your data store
Update your local database/data store with the cleaned data, following best practices for storing your cleaned data:

- Must maintain different instances / versions of data (raw and cleaned data)
- Must name the dataset files informatively
- Ensure both the raw and cleaned data is saved to your database/data store

In [ ]:
#FILL IN - saving data

## 5. Answer the research question

### **5.1:** Define and answer the research question 
Going back to the problem statement in step 1, use the cleaned data to answer the question you raised. Produce **at least** two visualizations using the cleaned data and explain how they help you answer the question.

*Research question:* FILL IN from answer to Step 1

In [ ]:
#Visual 1 - FILL IN

*Answer to research question:* FILL IN

In [ ]:
#Visual 2 - FILL IN

*Answer to research question:* FILL IN

### **5.2:** Reflection
In 2-4 sentences, if you had more time to complete the project, what actions would you take? For example, which data quality and structural issues would you look into further, and what research questions would you further explore?

*Answer:* FILL IN